# EDA — Additional Analysis
Extends da2.ipynb with: per-class AUC, feature importance, group distribution, class imbalance deep-dive, and a baseline comparison.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.multioutput import MultiOutputClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score

import os
os.makedirs('outputs', exist_ok=True)

SAVED = 'saved/'

X      = np.load(SAVED + 'X.npy',         allow_pickle=True)
Y      = np.load(SAVED + 'Y_encoded.npy', allow_pickle=True)
groups = np.load(SAVED + 'groups.npy',    allow_pickle=True)

# Try to load class names if available
try:
    classes = np.load(SAVED + 'classes.npy', allow_pickle=True)
except FileNotFoundError:
    classes = np.array([f'class_{i}' for i in range(Y.shape[1])])

print(f'X: {X.shape}  Y: {Y.shape}  groups: {groups.shape}')
print(f'Feature dim = {X.shape[1]}  |  Num classes = {Y.shape[1]}')

## 1. Class Imbalance Deep-Dive

In [ ]:
label_counts = Y.sum(axis=0)
label_freq   = label_counts / len(Y)

df_labels = pd.DataFrame({
    'class':     classes,
    'count':     label_counts.astype(int),
    'frequency': label_freq,
}).sort_values('count', ascending=False)

print('Top 10 most common classes:')
print(df_labels.head(10).to_string(index=False))
print('\nTop 10 rarest classes:')
print(df_labels.tail(10).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].bar(range(len(df_labels)), df_labels['count'], color='steelblue')
axes[0].set_xlabel('Class rank (sorted by frequency)')
axes[0].set_ylabel('Number of positive samples')
axes[0].set_title('Class frequency (sorted)')

axes[1].hist(label_counts, bins=40, color='salmon', edgecolor='white')
axes[1].set_xlabel('Positive samples per class')
axes[1].set_ylabel('Number of classes')
axes[1].set_title('Distribution of class frequencies')
axes[1].axvline(label_counts.mean(), color='red', linestyle='--', label=f'mean={label_counts.mean():.1f}')
axes[1].legend()

plt.tight_layout()
plt.savefig('outputs/eda_class_imbalance.png', dpi=150)
plt.show()

n_rare  = (label_counts < 10).sum()
n_empty = (label_counts == 0).sum()
print(f'\nClasses with < 10 samples: {n_rare}')
print(f'Classes with 0 samples:    {n_empty}')
print(f'Imbalance ratio (max/min non-zero): {label_counts[label_counts>0].max() / label_counts[label_counts>0].min():.1f}x')

## 2. Group / Recording Distribution

In [ ]:
unique_groups, group_counts = np.unique(groups, return_counts=True)
print(f'Number of unique recordings (groups): {len(unique_groups)}')
print(f'Segments per recording — min:{group_counts.min()}  max:{group_counts.max()}  mean:{group_counts.mean():.1f}')

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(group_counts, bins=30, color='mediumseagreen', edgecolor='white')
ax.set_xlabel('Number of 5-sec segments per recording')
ax.set_ylabel('Count')
ax.set_title('Segments per recording (GroupShuffleSplit groups)')
plt.tight_layout()
plt.savefig('outputs/eda_group_distribution.png', dpi=150)
plt.show()

# Confirm the split respects groups (no leakage)
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, Y, groups))
train_groups = set(groups[train_idx])
test_groups  = set(groups[test_idx])
overlap = train_groups & test_groups
print(f'\nTrain groups: {len(train_groups)}  |  Test groups: {len(test_groups)}')
print(f'Overlapping groups (should be 0): {len(overlap)}')

## 3. Baseline Comparison
Constant predictor (predict the training prior for every sample) gives a lower bound on AUC.

In [ ]:
X_train, X_test = X[train_idx], X[test_idx]
Y_train, Y_test = Y[train_idx], Y[test_idx]

# Baseline: predict class prior for every test sample
prior = Y_train.mean(axis=0)
Y_pred_baseline = np.tile(prior, (len(Y_test), 1))

baseline_aucs = []
for i in range(Y_test.shape[1]):
    if len(np.unique(Y_test[:, i])) > 1:
        baseline_aucs.append(roc_auc_score(Y_test[:, i], Y_pred_baseline[:, i]))

baseline_auc = np.mean(baseline_aucs) if baseline_aucs else 0.5
print(f'Baseline AUC (constant prior): {baseline_auc:.4f}')
print('(This should be ~0.5; if much higher, class distribution may be leaking)')

## 4. Feature Importance (Random Forest)
Which feature groups are most informative? Sums importance per block.

In [ ]:
# Use the most common class for a fast binary RF
label_id = int(np.argmax(Y_train.sum(axis=0)))
print(f'Training RF on class {label_id} ({classes[label_id]}) ...')

rf = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1)
rf.fit(X_train, Y_train[:, label_id])

importances = rf.feature_importances_

# Adjust these offsets to match your actual feature config in audioCharge.py
FEATURE_BLOCKS = [
    ('MFCC (mean+std)',       26),
    ('MFCC delta',            26),
    ('MFCC delta2',           26),
    ('Chroma',                24),
    ('Mel spectrogram',      256),
    ('Spectral centroid',      2),
    ('Spectral bandwidth',     2),
    ('Spectral rolloff',       2),
    ('Spectral contrast',     14),
    ('ZCR',                    2),
    ('RMS',                    2),
    ('Tonnetz',               12),
]

block_names, block_imp = [], []
idx = 0
for name, dim in FEATURE_BLOCKS:
    if idx + dim <= len(importances):
        block_names.append(name)
        block_imp.append(importances[idx:idx+dim].sum())
        idx += dim

df_imp = pd.DataFrame({'feature': block_names, 'importance': block_imp})
df_imp = df_imp.sort_values('importance', ascending=True)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(df_imp['feature'], df_imp['importance'], color='cornflowerblue')
ax.bar_label(bars, fmt='%.3f', padding=3, fontsize=9)
ax.set_xlabel('Summed feature importance')
ax.set_title(f'Feature group importance — RF on class "{classes[label_id]}"')
plt.tight_layout()
plt.savefig('outputs/eda_feature_importance.png', dpi=150)
plt.show()

## 5. Per-Class AUC Breakdown
Which species does the best model struggle with?

In [ ]:
scaler = StandardScaler()
Xtr = scaler.fit_transform(X_train)
Xte = scaler.transform(X_test)

valid = [i for i in range(Y_train.shape[1]) if len(np.unique(Y_train[:, i])) > 1]

lr = MultiOutputClassifier(LogisticRegression(C=0.01, max_iter=10000), n_jobs=-1)
lr.fit(Xtr, Y_train[:, valid])

proba_list = lr.predict_proba(Xte)
Y_pred_v = np.stack([
    p[:, 1] if p.shape[1] > 1 else np.zeros(p.shape[0])
    for p in proba_list
], axis=1)

per_class_auc = {}
for j, i in enumerate(valid):
    if len(np.unique(Y_test[:, i])) > 1:
        per_class_auc[classes[i]] = roc_auc_score(Y_test[:, i], Y_pred_v[:, j])

df_auc = pd.DataFrame(list(per_class_auc.items()), columns=['class', 'auc'])
df_auc = df_auc.sort_values('auc')

print(f'Evaluable classes: {len(df_auc)}/{Y.shape[1]}')
print('\n10 hardest classes:')
print(df_auc.head(10).to_string(index=False))
print('\n10 easiest classes:')
print(df_auc.tail(10).to_string(index=False))

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(range(len(df_auc)), df_auc['auc'].values, color='steelblue')
ax.axhline(0.5, color='red', linestyle='--', label='random baseline')
ax.axhline(df_auc['auc'].mean(), color='orange', linestyle='--', label=f'mean AUC={df_auc["auc"].mean():.3f}')
ax.set_xlabel('Class rank (sorted by AUC)')
ax.set_ylabel('AUC-ROC')
ax.set_title('Per-class AUC — Logistic Regression (best params)')
ax.legend()
plt.tight_layout()
plt.savefig('outputs/eda_per_class_auc.png', dpi=150)
plt.show()

## 6. Rare Classes vs AUC
Do rare classes systematically get lower AUC?

In [ ]:
df_auc['count'] = [
    int(Y_train[:, np.where(classes == c)[0][0]].sum()) if c in classes else 0
    for c in df_auc['class']
]

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(df_auc['count'], df_auc['auc'], alpha=0.6, color='steelblue')
ax.set_xlabel('Training samples for class')
ax.set_ylabel('AUC-ROC')
ax.set_title('AUC vs class frequency')
ax.axhline(0.5, color='red', linestyle='--', label='random')
ax.legend()
plt.tight_layout()
plt.savefig('outputs/eda_auc_vs_frequency.png', dpi=150)
plt.show()

corr = df_auc[['count','auc']].corr().iloc[0,1]
print(f'Pearson correlation (count vs AUC): {corr:.3f}')
print('(Positive = rarer classes are harder, as expected)')